# Creating Chroma DB with TestData

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("Testing.ipynb").resolve().parent.parent))
# This path is needed to properly import the modules and functions from the src directory.

from src.embedding.embeddings import get_embedding_function_local

In [18]:

import os
from dotenv import load_dotenv
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [4]:
# load_dotenv()

In [5]:
# Paths need to written relative to this notebook
chroma_path = Path("../chroma")
test_data_path = Path("../data/test")

test_path = test_data_path.resolve()
print(test_path)
print(list(os.listdir(test_path)))

C:\Users\Silas\Documents\03_Scripting\00_Git\Studium\writers-guide\data\test
['Understanding_Climate_Change.pdf']


In [6]:
def load_documents():
    # PDF File directory loader
    document_loader = PyPDFDirectoryLoader(test_data_path)
    return document_loader.load()

In [7]:
documents = load_documents()
documents[0:5]

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '..\\data\\test\\Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperature, \nprecipitation, and wind patterns, over an extended period. Over the past century, human \nactivities, particularly the burning of fossil fuels and deforestation, have significantly \ncontributed to climate change. \nHistorical Context \nThe Earth\'s climate has changed throughout history. Over the past 650,000 years, there have \nbeen seven cycles of glacial advance and retreat, with the abrupt end of the last ice age 

In [8]:
def split_documents(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)

In [9]:
chunks = split_documents(documents)
chunks[0:5]

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '..\\data\\test\\Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperature, \nprecipitation, and wind patterns, over an extended period. Over the past century, human \nactivities, particularly the burning of fossil fuels and deforestation, have significantly \ncontributed to climate change. \nHistorical Context \nThe Earth\'s climate has changed throughout history. Over the past 650,000 years, there have \nbeen seven cycles of glacial advance and retreat, with the abrupt end of the last ice age 

In [ ]:
def add_to_chroma_without_respect_to_id(chunks: list[Document]):
    abs_chroma_path = os.path.abspath(chroma_path)
    db = Chroma.from_documents(
        documents=chunks,
        embedding=get_embedding_function_local(),
        persist_directory=abs_chroma_path,
        # collection_name="fiction" # Organize collections to seperate tasks
    )

    print(f"Successfully saved {len(chunks)} chunks to {chroma_path}")

    # Add documents.
    existing_items = db.get(include=[])  # IDs are always included by default
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    new_chunks = []
    for chunk in chunks:
        new_chunks.append(chunk)

    if len(new_chunks):
        print(f"Adding new documents: {len(new_chunks)}")
        db.add_documents(new_chunks) 
        # db.persist()
    else:
        print("No new documents to add")

In [11]:
# Id Format: "data/testing/Understanding_Climate_Change.pdf:6:2"
# Page Source : Page Number : Chunk Index
def calculate_chunk_ids(chunks):
    # This will create IDs like 

    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"

        # If the page ID is the same as the last one, increment the index.
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id

        # Add it to the page meta-data.
        chunk.metadata["id"] = chunk_id

    return chunks

In [ ]:
def add_to_chroma_with_respect_to_id(chunks: list[Document]):
    abs_chroma_path = os.path.abspath(chroma_path)
    db = Chroma.from_documents(
        documents=chunks,
        embedding=get_embedding_function_local(),
        persist_directory=abs_chroma_path,
        # collection_name="fiction" # Organize collections to seperate tasks
    )
    print(f"Successfully saved {len(chunks)} chunks to {chroma_path}")


    # Add or Update the documents.
    existing_items = db.get(include=[])  # IDs are always included by default
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    
    # Only add documents that don't exist in the DB.
    new_chunks = []    
    chunks_with_ids = calculate_chunk_ids(chunks)
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    if len(new_chunks):
        print(f"Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)  # db.add_documents(new_chunks, ids=new_chunk_ids)
    else:
        print("No new documents to add")

In [13]:
add_to_chroma_with_respect_to_id(chunks)

Successfully saved 97 chunks to ..\chroma
Number of existing documents in DB: 97
Adding new documents: 97


In [14]:
# Main reference call to all the functions above
documents = load_documents()
chunks = split_documents(documents)
add_to_chroma_with_respect_to_id(chunks)


Successfully saved 97 chunks to ..\chroma
Number of existing documents in DB: 291
No new documents to add


In [ ]:
# Reset Chroma DB
# if os.path.exists(chroma_path):
#     shutil.rmtree(chroma_path)

# Check if the database should be cleared (using the --clear flag).
# parser = argparse.ArgumentParser()
# parser.add_argument("--reset", action="store_true", help="Reset the database.")
# args = parser.parse_args()
# if args.reset:
#     print("Clearing Database")
#     if os.path.exists(CHROMA_PATH):
#         shutil.rmtree(CHROMA_PATH)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: '..\\chroma\\370d3428-0f30-4357-bdd4-da1bd1e05e2d\\data_level0.bin'